# US Attraction EDA

In [15]:
import pandas as pd
import psycopg2
import psycopg2.extras as extras
import math
import openai 
from pgvector.psycopg2 import register_vector
import os

In [2]:
openai.api_key = os.getenv("OPENAI_API_KEY")

In [16]:
df = pd.read_csv("../data/cleaned_data_USA.csv", index_col=0)

In [25]:
df.head()

,name,main_category,rating,reviews,categories,address,city,country,state,zipcode,broader_category,Weighted_Score,Weighted_Average,All_Cities
0,Forsyth Park,Park,4.8,16538.0,"Park, Tourist attraction","Forsyth Park, Savannah, GA 31401",Savannah,USA,GA,NaN,Nature,79382.4,4.67,"Atlanta, Augusta, Chattanooga, Savannah"
1,The Cathedral Basilica of St. John the Baptist,Catholic cathedral,4.8,5911.0,"Catholic cathedral, Catholic church, Tourist a...",The Cathedral Basilica of St. John the Baptist...,Savannah,USA,GA,NaN,Religious,28372.8,4.80,"Atlanta, Augusta, Chattanooga, Savannah"
2,Fort Pulaski National Monument,Monument,4.8,5221.0,"Monument, Historical place, Historical landmar...","Fort Pulaski National Monument, 101 Fort Pulas...",Savannah,USA,GA,NaN,Cultural,25060.8,4.53,"Atlanta, Augusta, Chattanooga, Savannah"
3,Fountain at Forsyth Park,Historical landmark,4.8,4234.0,"Historical landmark, Tourist attraction","Fountain at Forsyth Park, 1 W Gaston St, Savan...",Savannah,USA,GA,NaN,Cultural,20323.2,4.53,"Atlanta, Augusta, Chattanooga, Savannah"
4,Wormsloe State Historic Site,Historical place museum,4.5,3615.0,"Historical place museum, Museum, Park, State park","Wormsloe State Historic Site, 7601 Skidaway Rd...",Savannah,USA,GA,NaN,Cultural,16267.5,4.53,"Atlanta, Augusta, Chattanooga, Savannah"


In [47]:
conn = psycopg2.connect(
    dbname="postgresdb",
    user="postgres",
    password="postgres_password",
    host="host.docker.internal",  # e.g., "localhost"
    port="5433"        # default PostgreSQL port
)

cursor = conn.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone())


('PostgreSQL 16.10 (Debian 16.10-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit',)


In [18]:
df = df.reset_index(names=["id"])
df.head()

,id,name,main_category,rating,reviews,categories,address,city,country,state,zipcode,broader_category,Weighted_Score,Weighted_Average,All_Cities
0,0,Forsyth Park,Park,4.8,16538.0,"Park, Tourist attraction","Forsyth Park, Savannah, GA 31401",Savannah,USA,GA,NaN,Nature,79382.4,4.67,"Atlanta, Augusta, Chattanooga, Savannah"
1,1,The Cathedral Basilica of St. John the Baptist,Catholic cathedral,4.8,5911.0,"Catholic cathedral, Catholic church, Tourist a...",The Cathedral Basilica of St. John the Baptist...,Savannah,USA,GA,NaN,Religious,28372.8,4.80,"Atlanta, Augusta, Chattanooga, Savannah"
2,2,Fort Pulaski National Monument,Monument,4.8,5221.0,"Monument, Historical place, Historical landmar...","Fort Pulaski National Monument, 101 Fort Pulas...",Savannah,USA,GA,NaN,Cultural,25060.8,4.53,"Atlanta, Augusta, Chattanooga, Savannah"
3,3,Fountain at Forsyth Park,Historical landmark,4.8,4234.0,"Historical landmark, Tourist attraction","Fountain at Forsyth Park, 1 W Gaston St, Savan...",Savannah,USA,GA,NaN,Cultural,20323.2,4.53,"Atlanta, Augusta, Chattanooga, Savannah"
4,4,Wormsloe State Historic Site,Historical place museum,4.5,3615.0,"Historical place museum, Museum, Park, State park","Wormsloe State Historic Site, 7601 Skidaway Rd...",Savannah,USA,GA,NaN,Cultural,16267.5,4.53,"Atlanta, Augusta, Chattanooga, Savannah"


In [28]:
df.dtypes

id                    int64
name                 object
main_category        object
rating              float64
reviews             float64
categories           object
address              object
city                 object
country              object
state                object
zipcode             float64
broader_category     object
Weighted_Score      float64
Weighted_Average    float64
All_Cities           object
dtype: object

# Postgres Upload

In [24]:
# Create table SQL – example with basic types to match DataFrame columns
create_table_query = '''
CREATE TABLE IF NOT EXISTS us_attractions (
    id INTEGER PRIMARY KEY,
    name VARCHAR(250),
    main_category VARCHAR(250),
    rating REAL,
    reviews REAL,
    categories VARCHAR(250),
    address VARCHAR(250),
    city VARCHAR(250),
    country VARCHAR(250),
    state VARCHAR(250),
    zipcode INTEGER,
    broader_category VARCHAR(250),
    weighted_score REAL,
    weighted_average REAL,
    all_cities VARCHAR(250)
);
'''
cursor.execute(create_table_query)
conn.commit()

In [9]:
len(df.columns)

14

In [25]:
def clean_tuple_for_insert(tup):
    return tuple(None if (isinstance(x, float) and math.isnan(x)) else x for x in tup)


def load_values(conn, df, table):
    tuples = [clean_tuple_for_insert(tuple(x)) for x in df.to_numpy()]
    col_names = [s.lower() for s in df.columns]
    cols = ','.join(col_names)
    query = "INSERT INTO %s(%s) VALUES %%s" % (table, cols)
    cursor = conn.cursor()
    try:
        extras.execute_values(cursor, query, tuples)
        conn.commit()
    except (Exception, psycopg2.DatabaseError) as error:
        print("Error: %s" % error)
        conn.rollback()
        cursor.close()
        return 1
    cursor.close()

In [26]:
load_values(conn, df, 'us_attractions')

In [30]:
# Create table SQL – example with basic types to match DataFrame columns
create_embedding_column = '''
ALTER TABLE us_attractions ADD COLUMN embedding vector(1536);
'''
cursor.execute(create_embedding_column)
conn.commit()
cursor.close()

In [56]:
## Create vector embeddings of existing records

register_vector(conn)
cur = conn.cursor()

# Fetch records that need embedding
cur.execute("SELECT id, name, categories, address, country, broader_category FROM us_attractions")
rows = cur.fetchall()

for row in rows:
    id = row[0]
    text = ','.join(map(str, row[1:]))

    # Call OpenAI embedding API (example)
    response = openai.embeddings.create(
        input=[text],
        model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    
    # # Update record with embedding
    cur.execute(
        "UPDATE us_attractions SET embedding = %s WHERE id = %s ",
        (embedding, id)
    )
conn.commit()

In [48]:
row = rows[0]
id = row[0]
text = ','.join(map(str, row[1:]))
# print(text)
# Call OpenAI embedding API (example)
response = openai.embeddings.create(
    input=[text],
    model="text-embedding-3-small"
)
embedding = response.data[0].embedding

In [50]:
len(embedding)

1536

In [57]:
cursor.close()
conn.close()

# Batch Upload to OpenAI

In [48]:
import json
import time
import math

# Step 1: Fetch records needing embeddings
cur = conn.cursor()
cur.execute("SELECT id, name, categories, address, country, broader_category FROM us_attractions")
rows = cur.fetchall()
cur.close()


In [51]:
import json
import time
from openai import OpenAI

client = OpenAI()

# Step 2: Prepare batch lines for the batch API (concatenate columns as string)
def create_batch_file(rows_chunk, batch_file_name):
    batch_lines = []
    for row in rows_chunk:
        row_id = row[0]
        combined_text = ','.join([str(x) for x in row[1:] if x is not None])
        batch_lines.append(json.dumps({
            "custom_id": str(row_id),
            "method": "POST",
            "url": "/v1/embeddings",
            "body": {
                "model": "text-embedding-3-small",
                "input": combined_text
            }
        }))
    with open(batch_file_name, "w") as f:
        f.write("\n".join(batch_lines))

# Step 3: Upload batch file and create batch job
def upload_and_create_batch(batch_file_name):
    # Upload file with purpose "batch"
    with open(batch_file_name, "rb") as f:
        upload_response = client.files.create(
            file=f,
            purpose="batch"
        )
    file_id = upload_response.id

    # Create batch job pointing to uploaded file
    batch_response = client.batches.create(
        input_file_id=file_id,
        endpoint="/v1/embeddings",
        completion_window="24h"
    )
    return batch_response.id

# Step 4: Poll batch job status until done
def wait_for_batch_completion(batch_job_id):
    while True:
        batch_status = client.batches.retrieve(batch_job_id)  # positional arg
        print(f"Batch status: {batch_status.status}")
        if batch_status.status == "completed":
            return batch_status
        elif batch_status.status == "failed":
            raise Exception(f"Batch job {batch_job_id} failed.")
        time.sleep(30)

# Step 5: Download results and update DB embeddings
def process_results(batch_status, cur, conn):
    result_file_id = batch_status.output_file_id
    # Retrieve the content of the file as bytes
    response_content = client.files.content(result_file_id)
    # Read all bytes from the response content
    results_content_bytes = response_content.read()  
    # Decode bytes to string
    results_content = results_content_bytes.decode("utf-8")

    for line in results_content.strip().split("\n"):
        record = json.loads(line)
        row_id = int(record["custom_id"])
        embedding = record["body"]["data"][0]["embedding"]
        cur.execute(
            "UPDATE us_attractions SET embedding = %s WHERE id = %s",
            (embedding, row_id)
        )
    conn.commit()



In [ ]:
# Main batch processing loop:
cur = conn.cursor()
rows_per_batch = 1000
num_batches = math.ceil(len(rows) / rows_per_batch)

for i in range(num_batches):
    start = i * rows_per_batch
    end = start + rows_per_batch
    rows_chunk = rows[start:end]

    batch_file = f"batch_input_{i+1}.jsonl"
    create_batch_file(rows_chunk, batch_file)

    batch_job_id = upload_and_create_batch(batch_file)
    print(f"Submitted batch job {batch_job_id} for batch {i+1}/{num_batches}")

    batch_status = wait_for_batch_completion(batch_job_id)
    print(f"Batch job {batch_job_id} completed. Processing results...")

    process_results(batch_status, cur, conn)
    print(f"Batch {i+1} processed and DB updated.")

    if i < num_batches - 1:
        print("Sleeping 60 seconds before next batch to avoid rate limits.")
        time.sleep(60)

cur.close()

In [55]:
rows_per_batch = 1000
num_batches = math.ceil(len(rows) / rows_per_batch)

In [ ]:
# Main batch processing loop:
cur = conn.cursor()


for i in range(num_batches):
    start = i * rows_per_batch
    end = start + rows_per_batch
    rows_chunk = rows[start:end]

    batch_file = f"batch_input_{i+1}.jsonl"
    create_batch_file(rows_chunk, batch_file)

    batch_job_id = upload_and_create_batch(batch_file)
    print(f"Submitted batch job {batch_job_id} for batch {i+1}/{num_batches}")

    batch_status = wait_for_batch_completion(batch_job_id)
    print(f"Batch job {batch_job_id} completed. Processing results...")

    process_results(batch_status, cur, conn)
    print(f"Batch {i+1} processed and DB updated.")

    if i < num_batches - 1:
        print("Sleeping 60 seconds before next batch to avoid rate limits.")
        time.sleep(60)

cur.close()

Download batch results from openAI and manually process them

In [ ]:
import json

cur = conn.cursor()

for i in range(1, num_batches+1):
    with open(f"../data/embeddings/batch_{i}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            row_id = int(data['custom_id'])
            embedding = data["response"]["body"]["data"][0]["embedding"]

            cur.execute(
                "UPDATE us_attractions SET embedding = %s WHERE id = %s ",
                (embedding, row_id)
            )
            # print(data[:2])  # Process the JSON object as needed
conn.commit()
cur.close()